# 分段运行 Embryo workflow

每个任务独立执行：自动分割、训练、分类、数邻居、绘制已有计数分布。先运行环境和配置单元，再将需要的 `RUN_...` 设为 `True`，只运行对应任务单元。默认所有任务关闭。

输入仅作读取，所有产物写入 `outputs/staged_notebook`。原运行 notebook 的内容和已有结果保留。

In [ ]:
from pathlib import Path
import json
import sys

working = Path.cwd().resolve()
ROOT = next((path for path in [working, *working.parents]
             if (path / 'EmbryoAnalyser/final_analyser.ipynb').is_file()), None)
if ROOT is None:
    raise RuntimeError('请从 summer_project 或 EmbryoAnalyser 目录启动 notebook。')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Image
from EmbryoAnalyser.workflow import train_workflow, classify_workflow, distribution_workflow
from EmbryoAnalyser.neighbours import run_neighbours

print('项目：', ROOT)

from EmbryoAnalyser.segmentation import segmentation_workflow


## 选择任务与输入

`SEGMENT_INPUTS` 支持单 TIFF、多个 TIFF 或递归目录。分割使用已有 Cellpose 环境和 `cpsam_v2`，自动运行，不打开 GUI。

对于 raw Z-stack，显式设置 `Z_PROJECTION = 'max'`，按各通道最大强度投影；或设置 `Z_PLANE` 为从 0 开始的层号。二维 TIFF 保持不变。不要同时指定两者。当前邻居宏使用二维 ROI。

分类输入仍是每个胚胎一个 measurement CSV；分割产生的 ROI 不直接代替测量 CSV。已有模型可直接分类，不必运行训练。

In [ ]:
OUTPUT = ROOT / 'outputs/staged_notebook'
SEGMENT_INPUTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/Composite.tif']
CSV_INPUTS = [ROOT / 'dataset/raw_dataset/E-CadGFP']
TRAIN_INPUTS = [ROOT / 'dataset/raw_dataset/gap43-mCherry/train']
NEIGHBOUR_INPUTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/Composite.tif']
EXISTING_COUNTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/slow_neighbour_counting.csv']
MODEL_FILE = OUTPUT / 'training/models.joblib'
MODELS = 'both'  # 'rf' / 'svm' / 'both'; applies when training
CELLPOSE_PYTHON = None  # default: existing cellpose_env; can set full Python path
Z_PROJECTION = None  # for raw Z-stack use 'max', or set Z_PLANE
Z_PLANE = None  # zero-based plane index
DIAMETER = None  # keep the original Cellpose default
USE_GPU = False
SEGMENT_TIMEOUT = None
FIJI_TIMEOUT = None
RUN_SEGMENTATION = False
RUN_TRAINING = False
RUN_CLASSIFICATION = False
RUN_NEIGHBOURS = False
RUN_DISTRIBUTION = False


## 1. 自动分割（独立任务）

设 `RUN_SEGMENTATION=True` 并运行此单元。完成后显示每张图的 ROI 数量，并将输出图片目录设为本次会话的 `NEIGHBOUR_INPUTS`。此单元只分割，不运行其他任务。

下次会话直接把报告中的 `image_dir` 填入 `NEIGHBOUR_INPUTS`，即可使用已有 ROI 数邻居，无需再次分割。

In [ ]:
if RUN_SEGMENTATION:
    segmented = segmentation_workflow(
        SEGMENT_INPUTS, OUTPUT / 'segmentation', python_executable=CELLPOSE_PYTHON,
        z_projection=Z_PROJECTION, z_plane=Z_PLANE, diameter=DIAMETER,
        use_gpu=USE_GPU, timeout=SEGMENT_TIMEOUT)
    display(pd.DataFrame(segmented['samples'])[['input_image', 'n_rois', 'image_file', 'roi_file']])
    NEIGHBOUR_INPUTS = [Path(segmented['image_dir'])]
    print('可用于数邻居的目录：', segmented['image_dir'])
    print('分割报告：', segmented['report_json'])


## 2. 训练 RF / SVM（独立任务，可跳过）

首次使用需要训练；已有模型时直接指定 MODEL_FILE。

In [ ]:
if RUN_TRAINING:
    trained = train_workflow(TRAIN_INPUTS, OUTPUT / 'training', model=MODELS)
    MODEL_FILE = trained['model_file']
    print('模型：', MODEL_FILE)


## 3. CSV 分类（独立任务）

设 RUN_CLASSIFICATION=True。使用 MODEL_FILE 中保存的模型与预处理状态，不触发分割或 Fiji。

In [ ]:
if RUN_CLASSIFICATION:
    classified = classify_workflow(CSV_INPUTS, MODEL_FILE, OUTPUT / 'classification')
    display(classified['predictions'])
    print('分类 CSV：', classified['predictions_csv'])


## 4. 数邻居与展示分布（独立任务）

设 RUN_NEIGHBOURS=True；NEIGHBOUR_INPUTS 必须指向有配套 ROI ZIP 的图片或目录。运行原 Fiji 宏，完成后展示 Python 分布。

In [ ]:
if RUN_NEIGHBOURS:
    counted = run_neighbours(NEIGHBOUR_INPUTS, OUTPUT / 'neighbours', timeout=FIJI_TIMEOUT)
    display(pd.read_csv(counted['summary_csv']))
    for sample in counted['samples']:
        display(pd.read_csv(sample['distribution_csv']))
        display(Image(filename=sample['png']))
    print('邻居汇总：', counted['summary_csv'])


## 5. 只绘制已有计数（独立任务）

设 RUN_DISTRIBUTION=True。不运行分割或 Fiji；直接读取已有 slow_neighbour_counting.csv 并展示分布。

In [ ]:
if RUN_DISTRIBUTION:
    plotted = distribution_workflow(EXISTING_COUNTS, OUTPUT / 'distributions')
    for sample in plotted['samples']:
        display(pd.read_csv(sample['distribution_csv']))
        display(Image(filename=sample['png']))
    print('分布汇总：', plotted['summary_csv'])
